##  Loading the Cleaned CSV

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, stddev, count, when, sum as spark_sum,
    round as spark_round, max as spark_max, min as spark_min
)
from pyspark.sql.window import Window
import time

In [2]:
# ── Start SparkSession ──
spark = SparkSession.builder \
    .appName("LogiScale_Phase2_BigDataProcessing") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .getOrCreate()


In [3]:
# ── Load cleaned CSV from Phase 1 ──
df = spark.read.csv(
    "dataco_cleaned.csv",
    header=True,
    inferSchema=True
)

In [4]:
print(f"Rows loaded : {df.count():,}")
print(f"Columns     : {len(df.columns)}")
df.printSchema()

Rows loaded : 180,519
Columns     : 32
root
 |-- Type: string (nullable = true)
 |-- ata_days: integer (nullable = true)
 |-- eta_days: integer (nullable = true)
 |-- benefit_per_order: double (nullable = true)
 |-- sales_per_customer: double (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_risk: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- market: string (nullable = true)
 |-- order_city: string (nullable = true)
 |-- order_country: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- units_sold: integer (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- profit_per_order: double (nullable = true

 ## GroupBy Route → Average Delay 

In [6]:
print("\n" + "="*50)
print("  Average Delay by Route")
print("="*50)

df_route_delay = df.groupBy("route", "shipping_mode").agg(

    spark_round(avg("delay_days"),   2).alias("avg_delay_days"),
    spark_round(stddev("delay_days"),2).alias("stddev_delay_days"),
    spark_round(avg("ata_days"),     2).alias("avg_actual_days"),
    spark_round(avg("eta_days"),     2).alias("avg_scheduled_days"),

    count("*").alias("total_orders"),

    spark_round(
        count(when(col("delivery_flag") == "Late",    1)) /
        count("*") * 100, 1
    ).alias("late_pct"),

    spark_round(
        count(when(col("delivery_flag") == "On Time", 1)) /
        count("*") * 100, 1
    ).alias("on_time_pct"),

    spark_round(
        count(when(col("delivery_flag") == "Early",   1)) /
        count("*") * 100, 1
    ).alias("early_pct")
)

# Show worst routes first
df_route_delay.orderBy("avg_delay_days", ascending=False).show(15, truncate=False)


  Average Delay by Route
+---------------+-------------+--------------+-----------------+---------------+------------------+------------+--------+-----------+---------+
|route          |shipping_mode|avg_delay_days|stddev_delay_days|avg_actual_days|avg_scheduled_days|total_orders|late_pct|on_time_pct|early_pct|
+---------------+-------------+--------------+-----------------+---------------+------------------+------------+--------+-----------+---------+
|Central Asia   |Second Class |2.21          |1.22             |4.21           |2.0               |117         |90.6    |9.4        |0.0      |
|Central Africa |Second Class |2.13          |1.44             |4.13           |2.0               |297         |82.5    |17.5       |0.0      |
|Eastern Asia   |Second Class |2.12          |1.37             |4.12           |2.0               |1406        |83.8    |16.2       |0.0      |
|Western Europe |Second Class |2.06          |1.42             |4.06           |2.0               |5438       